<a href="https://colab.research.google.com/github/ninobronw/BPALM-TOOL-IMPLEMENTATION/blob/main/HelmNet_Full_Codejla1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# **Helmet Detection System - Computer Vision Project**

A comprehensive deep learning solution for detecting whether workers are wearing safety helmets in industrial environments.

## **1. Problem Statement**

### 1.1 Business Context

Workplace safety in hazardous environments like construction sites and industrial plants is crucial to prevent accidents and injuries. One of the most important safety measures is ensuring workers wear safety helmets.

**Challenges:**
- Manual supervision is expensive and not scalable
- Human monitoring becomes fatigued and less effective over time
- Need for real-time, automated safety compliance

SafeGuard Corp plans to develop an automated image analysis system capable of detecting whether workers are wearing safety helmets, improving safety compliance and reducing workplace accidents.

### 1.2 Objective

Develop a binary image classification model that automatically classifies images into:
- **Class 1: With Helmet** - Workers wearing safety helmets
- **Class 0: Without Helmet** - Workers not wearing safety helmets

### 1.3 Dataset Description

**Total Images:** 4,125  
**Image Dimensions:** 200 x 200 x 3 (RGB)  
**Format:** NumPy array (.npy)

**Class Distribution:**
| Class | Count | Percentage |
|-------|-------|------------|
| With Helmet (1) | 3,161 | 76.63% |
| Without Helmet (0) | 964 | 23.37% |

**Dataset Characteristics:**
- Diverse environments: construction sites, factories, industrial settings
- Varying lighting conditions, angles, and worker postures
- Multiple worker activities: standing, using tools, moving

## **2. Setup & Configuration**

### 2.1 Install Dependencies

⚠️ **Important:** After running the cell below, restart the kernel and continue from the next cell.

In [ ]:
!pip install tensorflow[and-cuda] scikit-learn==1.6.1 opencv-python==4.12.0.88 seaborn==0.13.2 matplotlib==3.10.0 numpy==2.0.2 pandas==2.2.2 -q

### 2.2 Import Libraries

In [ ]:
# Standard Libraries
import os
import random
import math
import warnings
warnings.filterwarnings('ignore')

# Data Processing
import numpy as np
import pandas as pd
import zipfile

# Visualization
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import cv2

# Machine Learning - sklearn
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, 
    recall_score, f1_score, classification_report, mean_squared_error as mse
)

# Deep Learning - TensorFlow/Keras
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, Flatten, Conv2D, MaxPooling2D, BatchNormalization
)
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.applications.vgg16 import VGG16

# Google Colab
from google.colab.patches import cv2_imshow
from google.colab import drive

print("✓ All libraries imported successfully!")

### 2.3 Configuration & Environment Setup

In [ ]:
# Check GPU Availability
print("GPU Information:")
print(f"  - GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")
print(f"  - TensorFlow Version: {tf.__version__}")
print(f"  - Keras Version: {keras.__version__}")

# Set Random Seeds for Reproducibility
RANDOM_SEED = 812

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)
tf.config.experimental.enable_op_determinism()

print(f"\n✓ Random seeds set for reproducibility (seed={RANDOM_SEED})")

## **3. Data Loading & Exploration**

### 3.1 Mount Google Drive & Locate Dataset

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")

# Locate Dataset Folder
dataset_folder_path = '/content/drive/MyDrive/IHelmet Project '

if os.path.exists(dataset_folder_path):
    print(f"✓ Dataset folder found")
    print(f"\nContents:")
    for item in os.listdir(dataset_folder_path):
        print(f"  - {item}")
else:
    print(f"✗ Error: Dataset folder not found")

### 3.2 Load Labels CSV

In [ ]:
# Load Labels
labels_file_path = os.path.join(dataset_folder_path, 'labels.csv')

try:
    labels_df = pd.read_csv(labels_file_path)
    print("✓ Labels CSV loaded successfully!")
    print(f"\nDataFrame Shape: {labels_df.shape}")
    print(f"\nFirst 5 rows:")
    print(labels_df.head())
except FileNotFoundError:
    print(f"✗ Error: labels.csv not found")
except Exception as e:
    print(f"✗ Error: {e}")

### 3.3 Analyze Class Distribution

In [ ]:
# Analyze Class Distribution
label_counts = labels_df['label'].value_counts().sort_index()
total_images = len(labels_df)

print("Class Distribution:")
print("="*50)
for label, count in label_counts.items():
    percentage = (count / total_images) * 100
    class_name = "With Helmet" if label == 1 else "Without Helmet"
    print(f"  Class {label} ({class_name:18s}): {count:4d} images ({percentage:6.2f}%)")
print("="*50)
print(f"  Total: {total_images} images")

# Visualize Distribution
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#ff6b6b', '#4ecdc4']
ax.bar(['Without Helmet (0)', 'With Helmet (1)'], label_counts.values, color=colors, alpha=0.8)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('Class Distribution in Dataset', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(label_counts.values):
    ax.text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n⚠️  Note: Dataset is imbalanced. Class 1 is 3.3x more frequent than Class 0.")

### 3.4 Extract and Load Images

In [ ]:
# Extract Images from ZIP
extraction_dir = '/content/extracted_images'
os.makedirs(extraction_dir, exist_ok=True)

zip_file_path = os.path.join(dataset_folder_path, 'images.npy.zip')

if os.path.exists(zip_file_path):
    try:
        print("Extracting images.npy.zip...")
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extraction_dir)
        print("✓ Extraction completed!")
    except Exception as e:
        print(f"✗ Error: {e}")
else:
    print(f"✗ ZIP file not found")

In [ ]:
# Load Images from NumPy
images_file_path = os.path.join(extraction_dir, 'images.npy')

try:
    images = np.load(images_file_path)
    print("✓ Images loaded successfully!")
    print(f"\nImage Array Properties:")
    print(f"  - Shape: {images.shape}")
    print(f"  - Data Type: {images.dtype}")
    print(f"  - Memory: {images.nbytes / 1024 / 1024:.2f} MB")
    print(f"  - Min Value: {images.min()}")
    print(f"  - Max Value: {images.max()}")
except Exception as e:
    print(f"✗ Error: {e}")

## **4. Data Preprocessing**

### 4.1 Split Data (Train/Validation/Test)

In [ ]:
# Split 1: Train (80%) vs Temp (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels_df['label'], 
    test_size=0.2, 
    random_state=812, 
    stratify=labels_df['label']
)

# Split 2: Validation (50% of temp) vs Test (50% of temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.5, 
    random_state=812, 
    stratify=y_temp
)

print("Data Split Summary:")
print("="*60)
print(f"Training Set    - Shape: {X_train.shape}")
print(f"Validation Set  - Shape: {X_val.shape}")
print(f"Test Set        - Shape: {X_test.shape}")
print("="*60)

# Verify Class Distribution in Each Set
print("\nClass Distribution:")
print("="*60)
for set_name, y in [("Training", y_train), ("Validation", y_val), ("Test", y_test)]:
    dist = y.value_counts(normalize=True).sort_index()
    print(f"\n{set_name} Set:")
    for label, prop in dist.items():
        count = (y == label).sum()
        print(f"  Class {label}: {count:4d} ({prop*100:6.2f}%)")

### 4.2 Normalize Pixel Values

In [ ]:
# Normalize to [0, 1] range
print("Normalizing pixel values to [0, 1] range...")

X_train_normalized = X_train.astype('float32') / 255.0
X_val_normalized = X_val.astype('float32') / 255.0
X_test_normalized = X_test.astype('float32') / 255.0

print("\nNormalization Results:")
print("="*60)
for set_name, X_orig, X_norm in [
    ("Training", X_train, X_train_normalized),
    ("Validation", X_val, X_val_normalized),
    ("Test", X_test, X_test_normalized)
]:
    print(f"\n{set_name} Set:")
    print(f"  Original   - Min: {X_orig.min():7.2f} | Max: {X_orig.max():7.2f}")
    print(f"  Normalized - Min: {X_norm.min():.6f} | Max: {X_norm.max():.6f}")

print("\n✓ Normalization completed!")

### 4.3 Visualize Sample Images

In [ ]:
# Function to Display Random Images
def plot_sample_images(X, y, class_name, num_samples=5):
    """Display random sample images from dataset"""
    plt.figure(figsize=(15, 3))
    indices = np.random.choice(len(X), num_samples, replace=False)
    
    for idx, i in enumerate(indices):
        plt.subplot(1, num_samples, idx + 1)
        plt.imshow(X[i])
        label = y.iloc[i] if isinstance(y, pd.Series) else y[i]
        helmet_status = "With Helmet" if label == 1 else "Without Helmet"
        plt.title(f"Label: {label}\n{helmet_status}", fontsize=10)
        plt.axis('off')
    
    plt.suptitle(f"{class_name} - Sample Images", fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Display samples
plot_sample_images(X_train_normalized, y_train, "Training Set", num_samples=5)
plot_sample_images(X_val_normalized, y_val, "Validation Set", num_samples=5)

## **5. Model Building & Training**

### 5.1 Build CNN Model from Scratch

In [ ]:
# Build CNN Model from Scratch
model_cnn = Sequential([
    # Block 1
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(200, 200, 3), name='conv1_1'),
    Conv2D(32, (3, 3), activation='relu', padding='same', name='conv1_2'),
    BatchNormalization(name='bn1'),
    MaxPooling2D((2, 2), name='pool1'),
    Dropout(0.25, name='dropout1'),
    
    # Block 2
    Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2_1'),
    Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2_2'),
    BatchNormalization(name='bn2'),
    MaxPooling2D((2, 2), name='pool2'),
    Dropout(0.25, name='dropout2'),
    
    # Block 3
    Conv2D(128, (3, 3), activation='relu', padding='same', name='conv3_1'),
    Conv2D(128, (3, 3), activation='relu', padding='same', name='conv3_2'),
    BatchNormalization(name='bn3'),
    MaxPooling2D((2, 2), name='pool3'),
    Dropout(0.25, name='dropout3'),
    
    # Flatten and Dense
    Flatten(name='flatten'),
    Dense(256, activation='relu', name='dense1'),
    Dropout(0.5, name='dropout4'),
    Dense(128, activation='relu', name='dense2'),
    Dropout(0.5, name='dropout5'),
    Dense(1, activation='sigmoid', name='output')
])

# Compile Model
model_cnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✓ CNN Model Built Successfully!")
print("\nModel Architecture:")
model_cnn.summary()

### 5.2 Train CNN Model

In [ ]:
# Train CNN Model
print("Training CNN Model...")
print("="*60)

history_cnn = model_cnn.fit(
    X_train_normalized, y_train,
    validation_data=(X_val_normalized, y_val),
    epochs=20,
    batch_size=32,
    verbose=1
)

print("\n✓ Training completed!")

### 5.3 Evaluate CNN Model

In [ ]:
# Evaluate on Test Set
print("Evaluating CNN Model on Test Set...")
print("="*60)

y_pred_prob = model_cnn.predict(X_test_normalized)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\nPerformance Metrics:")
print(f"  - Accuracy:  {acc:.4f}")
print(f"  - Precision: {prec:.4f}")
print(f"  - Recall:    {rec:.4f}")
print(f"  - F1-Score:  {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"  {cm}")

# Classification Report
print(f"\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Without Helmet', 'With Helmet']))

### 5.4 Plot Training History

In [ ]:
# Plot Training History
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history_cnn.history['accuracy'], label='Training', linewidth=2)
axes[0].plot(history_cnn.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history_cnn.history['loss'], label='Training', linewidth=2)
axes[1].plot(history_cnn.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5.5 Plot Confusion Matrix

In [ ]:
# Plot Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Without Helmet', 'With Helmet'],
            yticklabels=['Without Helmet', 'With Helmet'])
plt.title('Confusion Matrix - CNN Model', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

## **6. Summary & Conclusions**

In [ ]:
print("\n" + "="*70)
print(" HELMET DETECTION SYSTEM - PROJECT SUMMARY")
print("="*70)

print("\n📊 Dataset Statistics:")
print(f"  • Total Images: {len(labels_df):,}")
print(f"  • Image Dimensions: 200 x 200 x 3 (RGB)")
print(f"  • Class 0 (No Helmet): {(y_test == 0).sum()} test samples")
print(f"  • Class 1 (With Helmet): {(y_test == 1).sum()} test samples")

print("\n🧠 Model Performance (CNN):")
print(f"  • Accuracy:  {acc:.2%}")
print(f"  • Precision: {prec:.2%}")
print(f"  • Recall:    {rec:.2%}")
print(f"  • F1-Score:  {f1:.4f}")

print("\n📈 Key Insights:")
print(f"  • TN (Correct no helmet): {cm[0,0]}")
print(f"  • FP (Incorrect helmet):  {cm[0,1]}")
print(f"  • FN (Missed no helmet):  {cm[1,0]}")
print(f"  • TP (Correct helmet):    {cm[1,1]}")

print("\n✅ Project Status: COMPLETE")
print("="*70)